# Compact benchmark algorithm comparison

This notebook compares **Exhaustive, Greedy, Heuristic, Uniform, and Random** on one-dimensional and two-dimensional datasets generated from unfiltered random field draws.

- **Exhaustive** optimizes over every feasible allocation and every node-specific pair/location decision.
- **Greedy** allocates each sensor where it gives the largest average pairwise error reduction at a location. It builds one local minimum-$P_{ijmn}$ tree after allocation.
- **Heuristic** uses the allocation and beam-search procedure defined in the implementation.
- **Uniform** uses a near-uniform allocation and a seeded random pair-location tree.
- **Random** provides a seeded random allocation and seeded random pair-location tree.

All methods are evaluated using the final sequential misclassification probability under equal field priors.

In [ ]:
# Resolve the project whether Jupyter starts here or from the repository root.
import importlib
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working = Path.cwd().resolve()
PROJECT = working / "problem-2-min-error-seq" if (working / "problem-2-min-error-seq").is_dir() else working
SOURCE = PROJECT / "src"
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))

_analysis_module = importlib.reload(importlib.import_module("analysis"))
run_analysis = _analysis_module.run_analysis
_figures_module = importlib.reload(importlib.import_module("figures"))
save_figures = _figures_module.save_figures
summary_markdown = _figures_module.summary_markdown

## Experimental settings

`test` mode selects **90 cases per spatial dimension**, balanced across field family, $K$, and $M$. `full` mode uses all eligible compact cases. Exhaustive search is restricted by `EXACT_K_MAX`; the compact datasets contain $K\in\{3,4\}$ and $M\in\{3,4\}$.

The time limit applies independently to each method-budget evaluation. Checkpoint reuse is enabled so an interrupted run can continue without recomputing completed records. Progress is printed after every completed case, separately for each dataset.

In [ ]:
# Use "full" for the complete study after confirming the test run.
RUN_MODE = "test"
RESUME = True
TIME_LIMIT_SECONDS = 30.0
EXACT_K_MAX = 4

# These controls apply only to the heuristic method.
BEAM_WIDTH = 3
MAX_ITERATIONS = 30

# This notebook is intentionally tied to the compact datasets.
STUDY = "compact"

## Run the matched comparison

Each method is run on the same selected case and sensor-budget combinations. A record is written immediately after each method-budget evaluation. The summary retains only matched combinations completed successfully by every requested method.

In [ ]:
# Run or resume the compact comparison with the settings above.
import importlib
run_analysis = importlib.reload(importlib.import_module("analysis")).run_analysis
analysis = run_analysis(
    project=PROJECT,
    study=STUDY,
    mode=RUN_MODE,
    resume=RESUME,
    exact_k_max=EXACT_K_MAX,
    time_limit=TIME_LIMIT_SECONDS,
    beam_width=BEAM_WIDTH,
    max_iterations=MAX_ITERATIONS,
)

# Confirm that this notebook used both compact datasets and all five methods.
expected_datasets = {"compact_benchmark_1d", "compact_benchmark_2d"}
expected_methods = {"Exhaustive", "Greedy", "Heuristic", "Uniform", "Random"}
observed_datasets = {row["dataset"] for row in analysis["rows"]}
observed_methods = {row["method"] for row in analysis["rows"]}
assert observed_datasets == expected_datasets
assert observed_methods == expected_methods
assert {int(row["K"]) for row in analysis["rows"]} == {3, 4}
assert {int(row["M"]) for row in analysis["rows"]} == {3, 4}

case_counts = {
    dimension: len({
        (row["dataset"], row["case"])
        for row in analysis["rows"]
        if int(row["dimension"]) == dimension
    })
    for dimension in (1, 2)
}
if RUN_MODE == "test":
    assert min(case_counts.values()) >= 180

print("Verified datasets:", sorted(observed_datasets))
print("Verified methods:", sorted(observed_methods))
print("Unique cases by dimension:", case_counts)
print("Results directory:", analysis["output"])

## Completion and matched summary

The completion counts expose timeouts or missing results before aggregation. The table reports the matched case count, mean budget fraction, mean error probability, mean runtime, and error above Exhaustive where applicable.

In [ ]:
# Report method completion before displaying aggregate values.
from collections import Counter

completion = Counter((row["method"], row["status"]) for row in analysis["rows"])
print("Method-status counts:", completion)
display(Markdown(summary_markdown(analysis["summary"])))

## Comparison figures

The figures show **mean error probability versus mean budget fraction**, runtime distributions by $K$, and error distributions by field family. Runtime and field-family distributions use grouped box plots. All figures use the same matched samples as the table.

In [ ]:
# Save publication-quality PDF and PNG versions, then display each figure.
save_figures = importlib.reload(importlib.import_module("figures")).save_figures
if not analysis["summary"]:
    raise RuntimeError("No matched completed cases are available for plotting")

figures = save_figures(
    analysis["summary"],
    analysis["rows"],
    analysis["output"],
)
for name, figure in figures.items():
    display(Markdown(f"### {name.replace('_', ' ').title()}"))
    display(figure)
    plt.close(figure)

In [ ]:
from collections import Counter

print(Counter(
    (row["dimension"], row["K"])
    for row in analysis["rows"]
    if row["method"] == "Exhaustive"
))